In [ ]:
!pip install pyspark

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataMesh_Saude") \
    .getOrCreate()

print("Spark iniciado!")

In [ ]:
!pip install boto3
# Rodar em um ambiente com boto3 configurado (não é obrigatório para o exercício)
import boto3

s3 = boto3.client("s3")
bucket = "datamesh-saude-exemplo"

dominios = ["pacientes", "exames", "atendimentos", "vigilancia", "farmacia"]
camadas = ["bronze", "silver", "gold"]

# s3.create_bucket(Bucket=bucket)  # descomente se tiver permissão

for dominio in dominios:
    for camada in camadas:
        chave = f"{dominio}/{camada}/.keep"
        # s3.put_object(Bucket=bucket, Key=chave, Body=b"")
        print(f"Estrutura planejada: s3://{bucket}/{chave}")

In [ ]:
pacientes = [
    (1, "João Silva", 35, "M", "São Paulo", "SP"),
    (2, "Maria Souza", 28, "F", "Rio de Janeiro", "RJ"),
    (3, "Carlos Lima", 42, "M", "Belo Horizonte", "MG"),
    (4, "Ana Paula", 31, "F", "São Paulo", "SP"),
    (5, "Pedro Alves", 25, "M", "Curitiba", "PR"),
    (6, "Beatriz Reis", 54, "F", "São Paulo", "SP"),
    (7, "Rafael Costa", 19, "M", "Rio de Janeiro", "RJ"),
    (8, "Juliana Dias", 63, "F", "Curitiba", "PR"),
]

df_pacientes = spark.createDataFrame(
    pacientes,
    ["paciente_id", "nome", "idade", "sexo", "cidade", "estado"]
)

df_pacientes.show()

In [ ]:
data_product_pacientes = {
    "nome": "Patient Data Product",
    "dominio": "Pacientes",
    "owner": "Equipe de Cadastro",
    "descricao": "Informações cadastrais básicas dos pacientes.",
    "campos": ["paciente_id", "nome", "idade", "sexo", "cidade", "estado"],
    "atualizacao": "Diária",
    "qualidade": "paciente_id não pode ser nulo e deve ser único",
    "seguranca": "Dado pessoal sensível (LGPD) — nome não deve ser exposto fora do domínio",
    "acesso": "SQL / API",
    "consumidores": ["Vigilância Epidemiológica", "Atendimentos", "Farmácia"]
}


In [ ]:
exames = [
    (5001, 1, "Dengue", "2026-09-01", "POSITIVO"),
    (5002, 2, "COVID-19", "2026-09-01", "NEGATIVO"),
    (5003, 3, "Dengue", "2026-09-02", "POSITIVO"),
    (5004, 4, "Influenza", "2026-09-02", "POSITIVO"),
    (5005, 5, "Dengue", "2026-09-03", "NEGATIVO"),
    (5006, 6, "Dengue", "2026-09-03", "POSITIVO"),
    (5007, 7, "COVID-19", "2026-09-04", "POSITIVO"),
    (5008, 8, "Influenza", "2026-09-04", "NEGATIVO"),
    (5009, 1, "COVID-19", "2026-09-05", "NEGATIVO"),
    (5010, 6, "Dengue", "2026-09-06", "POSITIVO"),
]

df_exames = spark.createDataFrame(
    exames,
    ["exame_id", "paciente_id", "tipo_exame", "data_exame", "resultado"]
)

df_exames.show()

In [ ]:
data_product_exames = {
    "nome": "Lab Exam Data Product",
    "dominio": "Exames/Laboratório",
    "owner": "Equipe de Laboratório",
    "descricao": "Resultados de exames laboratoriais realizados na rede.",
    "campos": ["exame_id", "paciente_id", "tipo_exame", "data_exame", "resultado"],
    "atualizacao": "A cada 6 horas",
    "qualidade": "resultado deve ser POSITIVO ou NEGATIVO (sem valores nulos ou fora do domínio)",
    "seguranca": "Dado de saúde sensível (LGPD, dado sensível de saúde)",
    "acesso": "SQL / Streaming",
    "consumidores": ["Vigilância Epidemiológica"]
}

In [ ]:
atendimentos = [
    (9001, 1, "2026-09-01", "UBS Centro", "Consulta", "LEVE"),
    (9002, 3, "2026-09-02", "Hospital Municipal", "Internação", "GRAVE"),
    (9003, 4, "2026-09-02", "UBS Norte", "Consulta", "MODERADA"),
    (9004, 6, "2026-09-03", "Hospital Municipal", "Internação", "GRAVE"),
    (9005, 7, "2026-09-04", "UBS Centro", "Consulta", "MODERADA"),
    (9006, 6, "2026-09-06", "Hospital Municipal", "Internação", "GRAVE"),
]

df_atendimentos = spark.createDataFrame(
    atendimentos,
    ["atendimento_id", "paciente_id", "data_atendimento", "unidade_saude", "tipo_atendimento", "gravidade"]
)

df_atendimentos.show()

In [ ]:
data_product_atendimentos = {
    "nome": "Healthcare Visit Data Product",
    "dominio": "Atendimentos",
    "owner": "Equipe Hospitalar",
    "descricao": "Consultas e internações realizadas na rede de saúde.",
    "campos": ["atendimento_id", "paciente_id", "data_atendimento", "unidade_saude", "tipo_atendimento", "gravidade"],
    "atualizacao": "Diária",
    "qualidade": "gravidade deve ser LEVE, MODERADA ou GRAVE",
    "seguranca": "Dado sensível de saúde",
    "acesso": "SQL",
    "consumidores": ["Vigilância Epidemiológica"]
}

In [ ]:
notificacoes = [
    (1, 1, "Dengue", "2026-09-01", "São Paulo", "CONFIRMADO"),
    (2, 3, "Dengue", "2026-09-02", "Belo Horizonte", "CONFIRMADO"),
    (3, 4, "Influenza", "2026-09-02", "São Paulo", "CONFIRMADO"),
    (4, 6, "Dengue", "2026-09-03", "São Paulo", "CONFIRMADO"),
    (5, 7, "COVID-19", "2026-09-04", "Rio de Janeiro", "CONFIRMADO"),
    (6, 8, "Influenza", "2026-09-04", "Curitiba", "SUSPEITO"),
    (7, 6, "Dengue", "2026-09-06", "São Paulo", "CONFIRMADO"),
]

df_notificacoes = spark.createDataFrame(
    notificacoes,
    ["notificacao_id", "paciente_id", "doenca", "data_notificacao", "cidade", "status"]
)

df_notificacoes.show()

In [ ]:
data_product_vigilancia = {
    "nome": "Disease Notification Data Product",
    "dominio": "Vigilância Epidemiológica",
    "owner": "Equipe de Vigilância",
    "descricao": "Notificações compulsórias de doenças de interesse de saúde pública.",
    "campos": ["notificacao_id", "paciente_id", "doenca", "data_notificacao", "cidade", "status"],
    "atualizacao": "Tempo real",
    "qualidade": "status deve ser CONFIRMADO, SUSPEITO ou DESCARTADO",
    "seguranca": "Dado sensível de saúde pública",
    "acesso": "API / SQL",
    "consumidores": ["Secretaria de Saúde", "Ministério da Saúde", "Analytics"]
}

In [ ]:
dispensacoes = [
    (701, 1, "Soro para reidratação", "2026-09-01", 1),
    (702, 3, "Antitérmico", "2026-09-02", 2),
    (703, 6, "Antitérmico", "2026-09-03", 1),
    (704, 7, "Antiviral", "2026-09-04", 1),
]

df_farmacia = spark.createDataFrame(
    dispensacoes,
    ["dispensacao_id", "paciente_id", "medicamento", "data_dispensacao", "quantidade"]
)

df_farmacia.show()

In [ ]:
data_product_farmacia = {
    "nome": "Medication Dispensing Data Product",
    "dominio": "Farmácia",
    "owner": "Equipe de Farmácia",
    "descricao": "Medicamentos dispensados pela rede pública.",
    "campos": ["dispensacao_id", "paciente_id", "medicamento", "data_dispensacao", "quantidade"],
    "atualizacao": "Diária",
    "qualidade": "quantidade deve ser maior que zero",
    "seguranca": "Dado sensível de saúde",
    "acesso": "SQL",
    "consumidores": ["Vigilância Epidemiológica"]
}

In [ ]:
catalogo = spark.createDataFrame(
    [
        (data_product_pacientes["nome"], data_product_pacientes["dominio"], data_product_pacientes["owner"], data_product_pacientes["atualizacao"]),
        (data_product_exames["nome"], data_product_exames["dominio"], data_product_exames["owner"], data_product_exames["atualizacao"]),
        (data_product_atendimentos["nome"], data_product_atendimentos["dominio"], data_product_atendimentos["owner"], data_product_atendimentos["atualizacao"]),
        (data_product_vigilancia["nome"], data_product_vigilancia["dominio"], data_product_vigilancia["owner"], data_product_vigilancia["atualizacao"]),
        (data_product_farmacia["nome"], data_product_farmacia["dominio"], data_product_farmacia["owner"], data_product_farmacia["atualizacao"]),
    ],
    ["data_product", "dominio", "owner", "atualizacao"]
)

catalogo.show(truncate=False)

In [ ]:
def checar_qualidade(df, coluna_chave, nome_data_product):
    total = df.count()
    nulos = df.filter(df[coluna_chave].isNull()).count()
    print(f"[{nome_data_product}] Total de registros: {total}")
    print(f"[{nome_data_product}] Registros com '{coluna_chave}' nulo: {nulos}")
    if nulos > 0:
        print(f"{nome_data_product} viola a regra federada de qualidade!")
    else:
        print(f" {nome_data_product} está de acordo com a governança.")

checar_qualidade(df_pacientes, "paciente_id", "Pacientes")
checar_qualidade(df_exames, "resultado", "Exames")
checar_qualidade(df_notificacoes, "status", "Vigilância")

In [ ]:
from pyspark.sql.functions import concat, substring, lit

df_pacientes_mascarado = df_pacientes.withColumn(
    "nome_mascarado",
    concat(substring("nome", 1, 3), lit("***"))
)

df_pacientes_mascarado.select("paciente_id", "nome_mascarado", "cidade", "estado").show()

In [ ]:
df_exames.createOrReplaceTempView("exames")
df_notificacoes.createOrReplaceTempView("notificacoes")
df_atendimentos.createOrReplaceTempView("atendimentos")
df_pacientes.createOrReplaceTempView("pacientes")

# Quantos exames positivos existem por tipo de doença?
spark.sql("""
    SELECT tipo_exame, COUNT(*) AS positivos
    FROM exames
    WHERE resultado = 'POSITIVO'
    GROUP BY tipo_exame
    ORDER BY positivos DESC
""").show()

In [ ]:
spark.sql("""
    SELECT
        resultado,
        COUNT(*) AS total,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM exames), 1) AS percentual
    FROM exames
    GROUP BY resultado
""").show()

In [ ]:
spark.sql("""
    SELECT cidade, COUNT(*) AS casos_confirmados
    FROM notificacoes
    WHERE status = 'CONFIRMADO'
    GROUP BY cidade
    ORDER BY casos_confirmados DESC
""").show()

In [ ]:
from pyspark.sql.functions import sum as _sum, count as _count, when

# total de notificações confirmadas por cidade
df_casos_cidade = df_notificacoes \
    .filter(df_notificacoes.status == "CONFIRMADO") \
    .groupBy("cidade") \
    .agg(_count("notificacao_id").alias("casos_confirmados"))

# total de internações graves por cidade (join com pacientes para saber a cidade)
df_internacoes_cidade = df_atendimentos \
    .filter(df_atendimentos.tipo_atendimento == "Internação") \
    .join(df_pacientes, "paciente_id") \
    .groupBy("cidade") \
    .agg(_count("atendimento_id").alias("internacoes"))

# total de exames positivos por cidade
df_exames_cidade = df_exames \
    .filter(df_exames.resultado == "POSITIVO") \
    .join(df_pacientes, "paciente_id") \
    .groupBy("cidade") \
    .agg(_count("exame_id").alias("exames_positivos"))

df_vigilancia_cidade = df_casos_cidade \
    .join(df_internacoes_cidade, "cidade", "left") \
    .join(df_exames_cidade, "cidade", "left")

df_vigilancia_cidade.show()

In [ ]:
df_vigilancia_cidade = df_vigilancia_cidade.fillna({
    "casos_confirmados": 0,
    "internacoes": 0,
    "exames_positivos": 0
})

df_vigilancia_cidade.show()

In [ ]:
df_vigilancia_cidade = df_vigilancia_cidade.withColumn(
    "risco_surto",
    when(
        (df_vigilancia_cidade["casos_confirmados"] >= 3) |
        (df_vigilancia_cidade["internacoes"] >= 2),
        "ALTO"
    ).when(
        (df_vigilancia_cidade["casos_confirmados"] >= 1),
        "MÉDIO"
    ).otherwise("BAIXO")
)

df_vigilancia_cidade.orderBy(df_vigilancia_cidade["casos_confirmados"].desc()).show()